## Lesson 06 — Pandas Part I: Basic operations

In this lesson we will cover some basic features of [Pandas](https://pandas.pydata.org).

We will also revisit some commands we already learned to learn them a bit more in depth.

### Readings

* [_Data Loading, Storage, and File Formats_, by Wes McKinney](https://wesmckinney.com/book/accessing-data)

### Table of Contents

* [Series](#Series)
* [DataFrame](#DataFrame)
* [index, columns](#index,-columns)
* [dtypes, info, describe](#dtypes,-info,-describe)
* [read_csv](#read_csv)
* [head, tail](#head,-tail)
* [Indexing with bracket/dot notation, loc, iloc](#Indexing-with-bracket/dot-notation,-loc,-iloc)
* [transpose](#Transpose)
* [to_csv, to_excel](#to_csv,-to_excel)
* [to_datetime](#to_datetime)

In [1]:
import pandas as pd
import numpy as np

### Series

A [Series](https://pandas.pydata.org/docs/dev/reference/api/pandas.Series.html) is a one-dimensional array with axis labels.

In [2]:
my_list = ["cubs", "pirates", "giants", "yankees", "donkeys"]
my_list

['cubs', 'pirates', 'giants', 'yankees', 'donkeys']

We can convert a list directly into a pandas Series:

In [3]:
series_from_list = pd.Series(my_list)
series_from_list

0       cubs
1    pirates
2     giants
3    yankees
4    donkeys
dtype: object

Indexing a Series is similar to lists and arrays:

In [4]:
series_from_list[3]

'yankees'

You can create a pandas Series from a numpy array:

In [5]:
my_array = np.arange(5)
my_array

array([0, 1, 2, 3, 4])

In [6]:
series_from_array = pd.Series(my_array)
series_from_array

0    0
1    1
2    2
3    3
4    4
dtype: int64

Indexing supports lists, as well as slices and scalars:

In [7]:
series_from_array[[1, 3]]

1    1
3    3
dtype: int64

In [8]:
series_from_array[3:]

3    3
4    4
dtype: int64

In [9]:
series_from_array[1]

np.int64(1)

### DataFrame

DataFrames are the gold standard for data representation.

#### 2D array to DataFrame

You can create a DataFrame from a numpy matrix (or list of lists) by passing it as argument to the DataFrame constructor:

In [10]:
rng = np.random.default_rng(seed=42)
my_2d_array = rng.standard_normal((5, 5))
my_2d_array

array([[ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519],
       [-1.30217951,  0.1278404 , -0.31624259, -0.01680116, -0.85304393],
       [ 0.87939797,  0.77779194,  0.0660307 ,  1.12724121,  0.46750934],
       [-0.85929246,  0.36875078, -0.9588826 ,  0.8784503 , -0.04992591],
       [-0.18486236, -0.68092954,  1.22254134, -0.15452948, -0.42832782]])

In [11]:
pd.DataFrame(my_2d_array)

,0,1,2,3,4
0,0.304717,-1.039984,0.750451,0.940565,-1.951035
1,-1.302180,0.127840,-0.316243,-0.016801,-0.853044
2,0.879398,0.777792,0.066031,1.127241,0.467509
3,-0.859292,0.368751,-0.958883,0.878450,-0.049926
4,-0.184862,-0.680930,1.222541,-0.154529,-0.428328


We can set the index and columns labels when we create the DataFrame.

Note that we can combine positional arguments (data) and keyword arguments `(index, columns)` as long as positional arguments come first.

In [12]:
df_from_2d_array = pd.DataFrame(
    my_2d_array,
    index=["row1", "row2", "row3", "row4", "row5"],
    columns=["col1", "col2", "col3", "col4", "col5"],
)
df_from_2d_array

,col1,col2,col3,col4,col5
row1,0.304717,-1.039984,0.750451,0.940565,-1.951035
row2,-1.302180,0.127840,-0.316243,-0.016801,-0.853044
row3,0.879398,0.777792,0.066031,1.127241,0.467509
row4,-0.859292,0.368751,-0.958883,0.878450,-0.049926
row5,-0.184862,-0.680930,1.222541,-0.154529,-0.428328


Also note that new lines do not affect the way that the dataframe is generated.

The following example WILL NOT generate a dataframe with two rows

In [13]:
# fmt: off
mylist = [
    0, 2, 4,
    6, 8, 10,
]
# fmt: on

In [14]:
mylist

[0, 2, 4, 6, 8, 10]

In [15]:
pd.DataFrame(mylist)

,0
0,0
1,2
2,4
3,6
4,8
5,10


But the following example (list inside of a list) will:

In [16]:
my_other_list = [
    [0, 2, 4],
    [6, 8, 10],
]
pd.DataFrame(my_other_list)

,0,1,2
0,0,2,4
1,6,8,10


#### Converting multiple Series to a DataFrame

As you have seen, a Series is essentially a column or row from a DataFrame. This means you can join multiple series together, either horizontally or vertically, to form a DataFrame.

**Method 1**: getting data as a list of series will orient them as rows. This is typically not ideal.

In [17]:
x = pd.DataFrame(data=[series_from_list, series_from_array])
x

,0,1,2,3,4
0,cubs,pirates,giants,yankees,donkeys
1,0,1,2,3,4


Note that each column has dtype object (string). This is because pandas will try to find a common data type which matches both series, in this case `object`:

In [18]:
x.dtypes

0    object
1    object
2    object
3    object
4    object
dtype: object

In this example, to flip our DataFrame, we need to transpose the table - we'll see this again later in the lesson.

In [19]:
x = x.transpose()
x

,0,1
0,cubs,0
1,pirates,1
2,giants,2
3,yankees,3
4,donkeys,4


The transposed columns have kept dtype object - we'll see how to fix this below.

In [20]:
x.dtypes

0    object
1    object
dtype: object

**Method 2**: pass a list or Series as value of dictionary with this method.

We do not have to define the column names with an extra argument.

In [21]:
y = pd.DataFrame({"a": series_from_list, "b": series_from_array})
y

,a,b
0,cubs,0
1,pirates,1
2,giants,2
3,yankees,3
4,donkeys,4


Importing the data as columns gives us the correct dtypes - the dtypes from the Series will be inherited.

In [22]:
y.dtypes

a    object
b     int64
dtype: object

**Method 3**: use `pd.concat()` to combine series in column orientation:

In [23]:
df = pd.concat([series_from_list, series_from_array], axis=1)
df

,0,1
0,cubs,0
1,pirates,1
2,giants,2
3,yankees,3
4,donkeys,4


Again, importing the data as columns gives us the correct dtypes:

In [24]:
df.dtypes

0    object
1     int64
dtype: object

As a curiosity, you can try concat with `axis=0`:

In [25]:
df_axis0 = pd.concat([series_from_list, series_from_array], axis=0)
df_axis0

0       cubs
1    pirates
2     giants
3    yankees
4    donkeys
0          0
1          1
2          2
3          3
4          4
dtype: object

### index, columns

Our new DataFrame has two important properties `index` and `column`. They can be seen as Series representing the labels of your dataset.

In [26]:
df

,0,1
0,cubs,0
1,pirates,1
2,giants,2
3,yankees,3
4,donkeys,4


In [27]:
df.index

RangeIndex(start=0, stop=5, step=1)

In [28]:
df.columns

RangeIndex(start=0, stop=2, step=1)

To better label your DataFrame, you can rename columns and indexes to more appropriate names.
    
**Method 1**: set the index and columns names to an existing DataFrame by using a list.

In [29]:
df.index = ["a", "b", "c", "d", "e"]
df.columns = ["team", "number"]
df

,team,number
a,cubs,0
b,pirates,1
c,giants,2
d,yankees,3
e,donkeys,4


In this case, the list will automatically be converted to an `Index` with dtype `object`:

In [30]:
df.index

Index(['a', 'b', 'c', 'd', 'e'], dtype='object')

In [31]:
df.columns

Index(['team', 'number'], dtype='object')

When you assign to a new column name using bracket notation, it adds the column to the end of the DataFrame.

In [32]:
df["integers"] = [2, 3, 5, 8, 13]
df

,team,number,integers
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


You can use `df.insert()` instead (similar to how we did it with the lists) to add a new column at a specific position:

In [33]:
df.insert(0, "integers2", [2, 3, 5, 8, 13])
df

,integers2,team,number,integers
a,2,cubs,0,2
b,3,pirates,1,3
c,5,giants,2,5
d,8,yankees,3,8
e,13,donkeys,4,13


Use `df.drop()` to delete a column:

In [34]:
df.drop("integers2", axis=1, inplace=True)
df

,team,number,integers
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


**Method 2**: You can use a dictionary mapping to rename columns. Provide the direct conversion from old names to new names:

In [35]:
df.rename(columns={"integers": "fibonacci"}, inplace=True)
df

,team,number,fibonacci
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


If you only want to _reorder_ the columns, you can do so by passing a list of column labels in the desired order:

In [36]:
df[["fibonacci", "team", "number"]]

,fibonacci,team,number
a,2,cubs,0
b,3,pirates,1
c,5,giants,2
d,8,yankees,3
e,13,donkeys,4


### dtypes, info, describe

The `dtypes` property gives the datatype of each column:

In [37]:
df.dtypes

team         object
number        int64
fibonacci     int64
dtype: object

The `shape` property shows the dimensions of the DataFrame (exactly the same output as numpy):

In [38]:
df.shape

(5, 3)

The `info()` method gives information about the index, columns, and memory usage:

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, a to e
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   team       5 non-null      object
 1   number     5 non-null      int64 
 2   fibonacci  5 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 160.0+ bytes


Note the hint in the _memory usage_ row. This count is not exact because it can be quite expensive to compute for really big datasets (as the computation is done per item, rather than an estimation per column).

If you need the exact size, use `memory_usage="deep"` option:

In [40]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, a to e
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   team       5 non-null      object
 1   number     5 non-null      int64 
 2   fibonacci  5 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 606.0 bytes


The `describe()` method returns basic statistics for all numeric columns:

In [41]:
df.describe()

,number,fibonacci
count,5.000000,5.000000
mean,2.000000,6.200000
std,1.581139,4.438468
min,0.000000,2.000000
25%,1.000000,3.000000
50%,2.000000,5.000000
75%,3.000000,8.000000
max,4.000000,13.000000


### read_csv

By default, column headers are taken from the first row and row indexes are integers starting from zero:

In [42]:
df_sio = pd.read_csv("../data/employment-12211-9008_en.csv")

In [43]:
df_sio.head()

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013,186000,387000,573000


By default, `read_csv` will infer the dtypes from the data:

In [44]:
df_sio.dtypes

Variable    object
Sector      object
Year         int64
Female       int64
Male         int64
Total        int64
dtype: object

In [45]:
df_sio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Variable  231 non-null    object
 1   Sector    231 non-null    object
 2   Year      231 non-null    int64 
 3   Female    231 non-null    int64 
 4   Male      231 non-null    int64 
 5   Total     231 non-null    int64 
dtypes: int64(4), object(2)
memory usage: 11.0+ KB


In [46]:
df_sio.describe()

,Year,Female,Male,Total
count,231.000000,2.310000e+02,2.310000e+02,2.310000e+02
mean,2014.000000,8.893333e+05,1.027610e+06,1.916944e+06
std,3.169145,1.008617e+06,1.270668e+06,2.002257e+06
min,2009.000000,7.000000e+03,1.000000e+04,1.800000e+04
25%,2011.000000,1.710000e+05,2.405000e+05,3.250000e+05
50%,2014.000000,4.850000e+05,6.600000e+05,1.290000e+06
75%,2017.000000,1.161000e+06,1.260000e+06,2.590500e+06
max,2019.000000,4.314000e+06,5.831000e+06,8.010000e+06


#### Changing dtype on DataFrame creation

You can specify `dtype`, `index_col`, and `header` parameters on the `read_csv` function.

If you provide `dtype=object` to the `read_csv`, every single entry will be rendered as an object, meaning string.

In [47]:
df_sio2 = pd.read_csv("../data/employment-12211-9008_en.csv", dtype=object, index_col=None, header=0)

In [48]:
df_sio2.dtypes

Variable    object
Sector      object
Year        object
Female      object
Male        object
Total       object
dtype: object

For more control, you can specify dtypes per column using a dictionary:

In [49]:
df_sio3 = pd.read_csv(
    "../data/employment-12211-9008_en.csv",
    dtype={
        "Variable": str,
        "Sector": str,
        "Year": int,
        "Female": int,
        "Male": np.int32,
        "Total": np.float64,
    },
    index_col=None,
    header=0,
)

In [50]:
df_sio3.dtypes

Variable     object
Sector       object
Year          int64
Female        int64
Male          int32
Total       float64
dtype: object

**Note:** By default `np.int64` and `int` are the same. The defaults are good enough for the majority of applications.

#### Changing dtype of columns after DataFrame is created

Sometimes it's better to specify the dtype as `object` and convert to the correct types later, since pandas is already very good at converting these from the raw data.

In [51]:
df_sio2.dtypes

Variable    object
Sector      object
Year        object
Female      object
Male        object
Total       object
dtype: object

In [52]:
df_sio2["Year"]

0      2009
1      2010
2      2011
3      2012
4      2013
       ... 
226    2015
227    2016
228    2017
229    2018
230    2019
Name: Year, Length: 231, dtype: object

**Method 1**: use a list comprehension to convert one column:

In [53]:
df_sio2["Year"] = [int(x) for x in df_sio2["Year"]]

In [54]:
df_sio2["Year"]

0      2009
1      2010
2      2011
3      2012
4      2013
       ... 
226    2015
227    2016
228    2017
229    2018
230    2019
Name: Year, Length: 231, dtype: int64

**Method 2**: use `pd.to_numeric()` to convert and assign back to the column:

In [55]:
df_sio2["Total"] = pd.to_numeric(df_sio2["Total"])

In [56]:
df_sio2["Total"]

0      648000
1      637000
2      639000
3      612000
4      573000
        ...  
226     18000
227     19000
228     21000
229     22000
230     19000
Name: Total, Length: 231, dtype: int64

**Method 3**: use `apply(pd.to_numeric)` to convert multiple columns at once:

In [57]:
df_sio2[["Male", "Female"]] = df_sio2[["Male", "Female"]].apply(pd.to_numeric)

In [58]:
df_sio2[["Male", "Female"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Male    231 non-null    int64
 1   Female  231 non-null    int64
dtypes: int64(2)
memory usage: 3.7 KB


In [59]:
df_sio2.dtypes

Variable    object
Sector      object
Year         int64
Female       int64
Male         int64
Total        int64
dtype: object

### head, tail

You can pass a number to `head()` to change how many rows are displayed:

In [60]:
df_sio.head(3)

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011,213000,426000,639000


`tail()` works the same way, but shows the last rows:

In [61]:
df_sio.tail(3)

,Variable,Sector,Year,Female,Male,Total
228,WZ08-U,Extraterritorial organisations and bodies,2017,10000,11000,21000
229,WZ08-U,Extraterritorial organisations and bodies,2018,11000,11000,22000
230,WZ08-U,Extraterritorial organisations and bodies,2019,10000,10000,19000


If we display the whole DataFrame, only a limited number of first and last rows are shown:

In [62]:
df_sio

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013,186000,387000,573000
...,...,...,...,...,...,...
226,WZ08-U,Extraterritorial organisations and bodies,2015,8000,11000,18000
227,WZ08-U,Extraterritorial organisations and bodies,2016,8000,10000,19000
228,WZ08-U,Extraterritorial organisations and bodies,2017,10000,11000,21000
229,WZ08-U,Extraterritorial organisations and bodies,2018,11000,11000,22000


We can configure the display limits using `pd.set_option()`:

In [63]:
pd.set_option("display.min_rows", 15)
pd.set_option("display.max_rows", 15)
df_sio

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013,186000,387000,573000
5,WZ08-A,"Agriculture, forestry and fishing",2014,188000,383000,571000
6,WZ08-A,"Agriculture, forestry and fishing",2015,180000,382000,562000
...,...,...,...,...,...,...
224,WZ08-U,Extraterritorial organisations and bodies,2013,8000,12000,20000
225,WZ08-U,Extraterritorial organisations and bodies,2014,7000,13000,21000


Reset all display options afterward:

In [64]:
pd.reset_option("all", silent=True)

### Indexing with bracket/dot notation, loc, iloc

Pandas has three indexing methods:

* `[ ]` and `.` work on labels of columns
* `.loc` works on labels of indexes and columns. If the DataFrame does not have indexes, this will work similar to `iloc`.
* `.iloc` works on the positions of indexes and columns (so it only takes integers)

In [65]:
df

,team,number,fibonacci
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


#### brackets only: column by header

To get a column as a Series, use the column header in brackets:

In [66]:
df["team"]

a       cubs
b    pirates
c     giants
d    yankees
e    donkeys
Name: team, dtype: object

For multiple columns, put a list inside the brackets (so two sets of brackets):

In [67]:
df[["team", "number"]]

,team,number
a,cubs,0
b,pirates,1
c,giants,2
d,yankees,3
e,donkeys,4


Using a list with a single column name returns a DataFrame instead of a Series:

In [68]:
df[["team"]]

,team
a,cubs
b,pirates
c,giants
d,yankees
e,donkeys


#### dot-notation

If the column name contains only alphanumeric characters (including underscores), you can use dot notation instead of brackets and quotes:

**Note**: this dot-notation always returns a Series.

In [69]:
df.team

a       cubs
b    pirates
c     giants
d    yankees
e    donkeys
Name: team, dtype: object

#### loc: row by index

To get a row by its label, use `.loc` with the row index:

In [70]:
df

,team,number,fibonacci
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


In [71]:
df.loc["a"]

team         cubs
number          0
fibonacci       2
Name: a, dtype: object

For multiple rows, put a list of labels inside the brackets:

In [72]:
df.loc[["a", "d"]]

,team,number,fibonacci
a,cubs,0,2
d,yankees,3,8


The syntax is slightly different when the DataFrame has a multi-index:

In [73]:
df_multi_index = df.set_index([df.index, "team"])
df_multi_index

,,number,fibonacci
,team,,
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


Note that now the index is of type `MultiIndex`.

In [74]:
df_multi_index.index

MultiIndex([('a',    'cubs'),
            ('b', 'pirates'),
            ('c',  'giants'),
            ('d', 'yankees'),
            ('e', 'donkeys')],
           names=[None, 'team'])

For every `.loc` command, you will now have to provide multiple entries to locate the exact match:

In [75]:
df_multi_index.loc["a", "cubs"]

number       0
fibonacci    2
Name: (a, cubs), dtype: int64

In [76]:
df_multi_index.loc[[("a", "cubs"), ("d", "yankees")]]

,,number,fibonacci
,team,,
a,cubs,0,2
d,yankees,3,8


You can also modify individual entries if you know their exact location:

In [77]:
df_multi_index.loc[("a", "cubs"), "number"] = 42

In [78]:
df_multi_index.loc[("a", "cubs")]

number       42
fibonacci     2
Name: (a, cubs), dtype: int64

Be careful when swapping indexes: if you set another index on your DataFrame, the original one will be lost:

In [79]:
df

,team,number,fibonacci
a,cubs,0,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


In [80]:
df.set_index(["team"])

,number,fibonacci
team,,
cubs,0,2
pirates,1,3
giants,2,5
yankees,3,8
donkeys,4,13


You can get around it by resetting the index first (assigning to a new column), then setting a new index:

In [81]:
df.reset_index(names=["letter"]).set_index(["team"])

,letter,number,fibonacci
team,,,
cubs,a,0,2
pirates,b,1,3
giants,c,2,5
yankees,d,3,8
donkeys,e,4,13


#### iloc: row (or column) by position

To get a row by its position, use `.iloc` with the row number:

In [82]:
df.iloc[0]

team         cubs
number          0
fibonacci       2
Name: a, dtype: object

For multiple rows, put a list of positions inside the brackets:

In [83]:
df.iloc[[0, 3]]

,team,number,fibonacci
a,cubs,0,2
d,yankees,3,8


You can also pass a slice:

In [84]:
df.iloc[2:]

,team,number,fibonacci
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


`iloc` also works with columns:

In [85]:
df.iloc[:, [0, 2]]

,team,fibonacci
a,cubs,2
b,pirates,3
c,giants,5
d,yankees,8
e,donkeys,13


You can also modify individual entries if you know their exact position:

In [86]:
df.iloc[0, 1] = 42
df.iloc[0]

team         cubs
number         42
fibonacci       2
Name: a, dtype: object

### transpose

Transposing a DataFrame has the same effect as transposing a numpy array: the columns become the index, and the index becomes the columns.

In [87]:
df.transpose()

,a,b,c,d,e
team,cubs,pirates,giants,yankees,donkeys
number,42,1,2,3,4
fibonacci,2,3,5,8,13


Calling `df.T` has the same effect as transposing:

In [88]:
df.T

,a,b,c,d,e
team,cubs,pirates,giants,yankees,donkeys
number,42,1,2,3,4
fibonacci,2,3,5,8,13


### to_csv, to_excel

Once you are done with your data analysis task, you might want to export the data to store it on disk or to share it with your colleagues.

For standard applications, CSV is a good format: it is fast and efficient, easy to read, and allows easy compression. You can export it with `.to_csv()`.

For further analysis by team members, it can be useful to export it as Excel or other spreadsheet format. You can do this with `.to_excel()`.

Make sure you have the Excel dependencies installed:

In [89]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


Use `to_csv` with defaults (comma-separated):

In [90]:
df.to_csv("teams.csv")

Use the `sep` option if the separator is not a comma:

In [91]:
df.to_csv("teams.tsv", sep="\t")

You can also specify an index label:

In [92]:
df.to_csv("teams.csv", index_label="index")

`to_excel` writes to an Excel file (requires the `openpyxl` package):

In [93]:
df.to_excel("teams.xlsx", index_label="index")

### read_csv, read_excel

Read the CSV file back with default settings:

In [94]:
pd.read_csv("teams.csv")

,index,team,number,fibonacci
0,a,cubs,42,2
1,b,pirates,1,3
2,c,giants,2,5
3,d,yankees,3,8
4,e,donkeys,4,13


Specify the first column of the CSV as the index with `index_col`:

In [95]:
df1 = pd.read_csv("teams.csv", index_col=0)
df1

,team,number,fibonacci
index,,,
a,cubs,42,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


The default datatypes are inferred automatically:

In [96]:
df1.dtypes

team         object
number        int64
fibonacci     int64
dtype: object

We can also specify the dtypes when reading. Here, all columns are read as `object`:

In [97]:
df2 = pd.read_csv("teams.csv", index_col=0, dtype=object)
df3 = pd.read_csv("teams.csv", index_col=0, dtype={"team": object, "number": np.float16, "fibonacci": np.int64})

Verifying that all columns have dtype `object`:

In [98]:
df2.dtypes

team         object
number       object
fibonacci    object
dtype: object

When dtypes are specified per column, each column gets its own type:

In [99]:
df3.dtypes

team          object
number       float16
fibonacci      int64
dtype: object

Use the `sep` option to read tab-separated files:

In [100]:
df4 = pd.read_csv("teams.tsv", index_col=0, sep="\t")
df4

,team,number,fibonacci
a,cubs,42,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


`read_excel` reads Excel files (requires the `openpyxl` package):

In [101]:
df5 = pd.read_excel("teams.xlsx", index_col=0)
df5

,team,number,fibonacci
index,,,
a,cubs,42,2
b,pirates,1,3
c,giants,2,5
d,yankees,3,8
e,donkeys,4,13


**Note:** Excel formats can be **pretty wild**. Pandas requires the Excel files to be nicely formatted for reading. You also have the option to specify the Sheet.

If you have such an Excel file, additional options might help you. Refer to the [read_excel documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) for more information.

In [102]:
# cleanup created files
import pathlib

pathlib.Path("teams.csv").unlink(missing_ok=True)
pathlib.Path("teams.tsv").unlink(missing_ok=True)
pathlib.Path("teams.xlsx").unlink(missing_ok=True)

### to_datetime

We will cover time series in greater detail in a future lesson.

For now, just know you can convert datetimes using `pd.to_datetime` and using format templates.

In [103]:
df_sio.head()

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013,186000,387000,573000


In [104]:
df_sio.dtypes

Variable    object
Sector      object
Year         int64
Female       int64
Male         int64
Total        int64
dtype: object

Our `pd.to_datetime` will generate a new Series, but with the `datetime64` datatype:

In [105]:
time = pd.to_datetime(df_sio["Year"], format="%Y")
time.head()

0   2009-01-01
1   2010-01-01
2   2011-01-01
3   2012-01-01
4   2013-01-01
Name: Year, dtype: datetime64[ns]

We can assign this Series in place of our original column:

In [106]:
df_sio["Year"] = time

Notice that this changes how our dates are represented:

In [107]:
df_sio.head()

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009-01-01,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010-01-01,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011-01-01,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012-01-01,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013-01-01,186000,387000,573000


We will see that the main advantage of using a Timeseries is that we can re-sample our dataset easily.

If you look at the datatypes, they now show `datetime64[ns]`:

In [108]:
df_sio.dtypes

Variable            object
Sector              object
Year        datetime64[ns]
Female               int64
Male                 int64
Total                int64
dtype: object

To do this in a single step, we can use `read_csv`'s `parse_dates` keyword:

In [109]:
df_sio4 = pd.read_csv("../data/employment-12211-9008_en.csv", index_col=None, parse_dates=["Year"])

In [110]:
df_sio4.head()

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009-01-01,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010-01-01,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011-01-01,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012-01-01,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013-01-01,186000,387000,573000


In [111]:
df_sio4.dtypes

Variable            object
Sector              object
Year        datetime64[ns]
Female               int64
Male                 int64
Total                int64
dtype: object

For maximal control, `parse_dates` can be combined with `dtype`:

In [112]:
df_sio5 = pd.read_csv(
    "../data/employment-12211-9008_en.csv", index_col=None, parse_dates=["Year"], dtype={"Total": int}
).head()

In [113]:
df_sio5.head()

,Variable,Sector,Year,Female,Male,Total
0,WZ08-A,"Agriculture, forestry and fishing",2009-01-01,220000,428000,648000
1,WZ08-A,"Agriculture, forestry and fishing",2010-01-01,214000,423000,637000
2,WZ08-A,"Agriculture, forestry and fishing",2011-01-01,213000,426000,639000
3,WZ08-A,"Agriculture, forestry and fishing",2012-01-01,200000,413000,612000
4,WZ08-A,"Agriculture, forestry and fishing",2013-01-01,186000,387000,573000


In [114]:
df_sio5.dtypes

Variable            object
Sector              object
Year        datetime64[ns]
Female               int64
Male                 int64
Total                int64
dtype: object